# Silver 2 — build_carrier_invoice
Reads all carrier staging tables, normalises to a canonical schema, UNIONs into `Tables/silver/carrier_invoice`.

**Parameters injected by Airflow:** `entity`, `source_names` (JSON list), `env`

In [ ]:
# Fabric Notebook — Silver 2: Build unified carrier_invoice from all staging tables
# ==================================================================================
# Triggered by: medallion_pipeline DAG after all silver1 tasks for this entity succeed
# Parameters:
#   entity       : "carrier_invoice"
#   source_names : JSON list of source_name strings that feed this entity
#   env          : "dev" | "prod"
#
# Responsibility:
#   1. For each source_name, read its staging Delta table from Silver Lakehouse
#   2. Apply a common schema mapping so all carriers share the same column set
#   3. UNION ALL into a single Silver 2 Delta table: carrier_invoice
#   4. Log result

import json
import sys
from pathlib import Path

### Parameters

In [ ]:
try:
    entity = entity  # noqa: F821
    source_names_str = source_names  # noqa: F821
    env = env  # noqa: F821
except NameError:
    entity = "carrier_invoice"
    source_names_str = '["fedex_direct_invoice", "dhl_direct_invoice", "ups_carrier_report"]'
    env = "dev"

source_names: list[str] = json.loads(source_names_str)
silver2_table = entity  # e.g. "carrier_invoice"

### Shared lib

In [ ]:
sys.path.insert(0, "/lakehouse/default/Files/shared")
import shared_functions as gf

### Config

In [ ]:
config_path = Path(__file__).parent.parent.parent.parent / "config" / f"{env}.json"
cfg: dict = json.loads(config_path.read_text())

WORKSPACE   = cfg["workspace_name"]
LAKEHOUSE   = cfg["lakehouse"]
DW_ENDPOINT = cfg["gold_warehouse_sql_endpoint"]
DW_DATABASE = cfg["gold_warehouse"]

try:
    run_id = mssparkutils.notebook.run("_get_run_id")  # noqa: F821
except Exception:
    run_id = "local"

print(f"[silver2/build_carrier_invoice] entity={entity} sources={source_names}")

### Canonical Silver 2 schema for carrier_invoice

In [ ]:
# All carrier staging tables are mapped to these columns before UNION.
# Columns not present in a source are filled with NULL.
CANONICAL_COLUMNS = [
    "invoice_number",
    "invoice_date",
    "carrier",
    "purchase_contract",
    "tracking_number",
    "shipment_date",
    "origin_country",
    "destination_country",
    "weight_kg",
    "charge_amount",
    "charge_currency",
    "service_type",
    "account_number",
    "_source_name",
    "_source_file",
    "_run_id",
]

### Column mapping per carrier (add new carriers here)

In [ ]:
# Maps canonical_column → source_column per carrier prefix
COLUMN_MAP: dict[str, dict[str, str]] = {
    "fedex": {
        "invoice_number":    "invoice_no",
        "invoice_date":      "invoice_date",
        "carrier":           None,          # filled with literal "fedex"
        "tracking_number":   "tracking_id",
        "shipment_date":     "ship_date",
        "weight_kg":         "weight_lbs",  # needs conversion
        "charge_amount":     "net_charge",
        "charge_currency":   "currency",
        "service_type":      "service_code",
        "account_number":    "account_no",
    },
    "dhl": {
        "invoice_number":    "invoice_number",
        "invoice_date":      "billing_date",
        "carrier":           None,
        "tracking_number":   "waybill_no",
        "shipment_date":     "shipment_date",
        "weight_kg":         "weight_kg",
        "charge_amount":     "total_charge",
        "charge_currency":   "currency_code",
        "service_type":      "product_code",
        "account_number":    "shipper_account",
    },
    "ups": {
        "invoice_number":    "invoice_nbr",
        "invoice_date":      "invoice_dt",
        "carrier":           None,
        "tracking_number":   "pkg_tracking_no",
        "shipment_date":     "pickup_date",
        "weight_kg":         "billed_weight",
        "charge_amount":     "billed_amount",
        "charge_currency":   "currency",
        "service_type":      "service_description",
        "account_number":    "ups_account",
    },
}

LBS_TO_KG = 0.453592

from pyspark.sql import SparkSession  # noqa: E402
import pyspark.sql.functions as F     # noqa: E402

spark = SparkSession.builder.getOrCreate()
all_frames = []

### 1. Read each staging table and normalise to canonical schema

In [ ]:
for source_name in source_names:
    staging_table = f"staging_{source_name}"
    staging_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{staging_table}")

    print(f"  Reading {staging_table} ...")
    try:
        sdf = spark.read.format("delta").load(staging_abfss)
    except Exception as exc:
        raise RuntimeError(f"Cannot read {staging_table} — did silver1 succeed? {exc}") from exc

    # Determine carrier from source_name prefix
    carrier = source_name.split("_")[0]  # "fedex_direct_invoice" → "fedex"
    mapping = COLUMN_MAP.get(carrier, {})

    # Build select expressions for each canonical column
    select_exprs = []
    for canonical_col in CANONICAL_COLUMNS:
        if canonical_col == "carrier":
            select_exprs.append(F.lit(carrier).alias("carrier"))
        elif canonical_col == "purchase_contract":
            select_exprs.append(F.lit(source_name).alias("purchase_contract"))
        elif canonical_col.startswith("_"):
            if canonical_col in sdf.columns:
                select_exprs.append(F.col(canonical_col))
            else:
                select_exprs.append(F.lit(None).cast("string").alias(canonical_col))
        else:
            src_col = mapping.get(canonical_col)
            if src_col and src_col in sdf.columns:
                # Special unit conversions
                if canonical_col == "weight_kg" and carrier == "fedex":
                    select_exprs.append(
                        (F.col(src_col).cast("double") * F.lit(LBS_TO_KG)).alias("weight_kg")
                    )
                else:
                    select_exprs.append(F.col(src_col).cast("string").alias(canonical_col))
            else:
                select_exprs.append(F.lit(None).cast("string").alias(canonical_col))

    normalised = sdf.select(select_exprs)
    all_frames.append(normalised)
    print(f"    {staging_table}: {normalised.count()} rows")

if not all_frames:
    raise RuntimeError("No staging frames loaded")

### 2. UNION ALL

In [ ]:
from functools import reduce  # noqa: E402
from pyspark.sql import DataFrame  # noqa: E402

unified = reduce(DataFrame.unionAll, all_frames)
total_rows = unified.count()
print(f"  Unified rows: {total_rows}")

### 3. Write to Silver 2 Delta table

In [ ]:
silver2_abfss = gf.onelake_abfss(WORKSPACE, LAKEHOUSE, f"Tables/silver/{silver2_table}")

unified.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(silver2_abfss)
print(f"  Written to {silver2_abfss}")

### 4. Log

In [ ]:
with gf.dw_connection(DW_ENDPOINT, DW_DATABASE) as conn:
    gf.log_run(
        conn,
        source_name=entity,
        layer="silver2",
        status="success",
        rows_processed=total_rows,
        run_id=run_id,
    )

result = {"status": "success", "entity": entity, "rows": total_rows, "table": silver2_table}
print(f"  Result: {result}")
mssparkutils.notebook.exit(json.dumps(result))  # noqa: F821